# Resume Screening (ATS Scoring) with Groq LLM

In [ ]:
%pip install pymupdf python-docx python-dotenv openai ipykernel

In [ ]:
import os
import re
import json
import getpass

import pymupdf
from docx import Document
from openai import OpenAI
from dotenv import load_dotenv

## Step 1: Extract text from resume (PDF or DOCX)

In [ ]:
def extract_resume_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        doc = pymupdf.open(file_path)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()

    elif ext == ".docx":
        doc = Document(file_path)
        text = "\n".join(p.text for p in doc.paragraphs)

    else:
        raise ValueError("Unsupported file format")

    return text

In [ ]:
resume_path = "data/resumes/resume1.pdf"   # change to your file (.pdf or .docx)

print(os.path.exists(resume_path))

resume_text = extract_resume_text(resume_path)
print(resume_text[:500])

In [ ]:
print("Total characters:", len(resume_text))
print("Total words:", len(resume_text.split()))

## Step 2: Preprocessing

In [ ]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

cleaned_resume = clean_text(resume_text)

print("Original length:", len(resume_text))
print("Cleaned length:", len(cleaned_resume))

## Step 3: Groq LLM setup

In [ ]:
load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API Key: ")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-120b"

Helper function: sends a prompt and returns the answer as a Python dictionary.

In [ ]:
def ask_llm_json(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

## Step 4: Resume to structured data

In [ ]:
resume_prompt = f"""
Analyze the following resume.

Extract:
1. name
2. technical_skills (short names only, like "Python", "SQL", "PyTorch")
3. soft_skills
4. education
5. experience
6. projects
7. certifications

Return ONLY valid JSON.

Resume:
{cleaned_resume}
"""

resume_data = ask_llm_json(resume_prompt)

In [ ]:
print(resume_data["name"])
print(resume_data["technical_skills"])
print(resume_data["education"])

## Step 5: Job description to structured data

In [ ]:
job_description = """
Job Title: AI/ML Engineer

Requirements:
- Python programming
- Machine Learning
- Deep Learning
- TensorFlow or PyTorch
- NLP and Generative AI
- SQL
- Problem-solving skills
- Communication skills

Education:
Bachelor's degree in Computer Science or related field.

Experience:
0-2 years.

Responsibilities:
- Develop machine learning models.
- Build AI-powered applications.
- Work with NLP and LLM technologies.
- Analyze and preprocess datasets.
"""

In [ ]:
jd_prompt = f"""
Analyze the following job description.

Extract:
1. job_title
2. technical_skills (short names only, like "Python", "SQL", "PyTorch")
3. soft_skills
4. education
5. experience
6. responsibilities

Return ONLY valid JSON.

Job Description:
{job_description}
"""

job_data = ask_llm_json(jd_prompt)
print(job_data)

## Step 6: ATS scoring (skill match)

In [ ]:
resume_skills = [s.lower() for s in resume_data["technical_skills"]]
job_skills = [s.lower() for s in job_data["technical_skills"]]
resume_text_lower = cleaned_resume.lower()

matched_skills = []
missing_skills = []

for skill in job_skills:
    if skill in resume_skills or skill in resume_text_lower:
        matched_skills.append(skill)
    else:
        missing_skills.append(skill)

print("Matched Skills:", matched_skills)
print("Missing Skills:", missing_skills)

In [ ]:
ats_score = round(len(matched_skills) / len(job_skills) * 100, 1)

print("ATS Score:", ats_score, "%")

## Step 7: Improvement suggestions

In [ ]:
suggest_prompt = f"""
A candidate applied for the job: {job_data["job_title"]}.

Matched skills: {matched_skills}
Missing skills: {missing_skills}
ATS score: {ats_score}%

Give 3 short, practical tips to improve this resume for the job.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": suggest_prompt}],
    temperature=0
)

print(response.choices[0].message.content)